# N20 Pure-State Entanglement Slope Vs Cycle, Resumable

This Colab/A100 notebook runs the canonical GPU Markov circuit in completed sample blocks. Each block writes a full entropy-contour accumulator checkpoint only after all cycles finish, so interrupted Colab runtimes can resume from the last completed block.

The final merge writes the same cycle-campaign products as the original notebook: contour batches, entropy curves, slope-vs-cycle tables, figures, run summary, and campaign manifest.


## 1. Setup

In [ ]:
from __future__ import annotations

import json
from pathlib import Path
import sys
import time

try:
    import google.colab  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive  # type: ignore
    drive.mount('/content/drive', force_remount=False)

BUNDLE_NAME = 'colab_large_entanglement_scaling_N20'


def is_bundle_root(path: Path) -> bool:
    return (path / 'src' / 'classA_U1FGTN_gpu.py').exists() and (path / 'src' / 'strip_entropy_streaming_gpu.py').exists()


candidates = []
for base in [Path.cwd(), *Path.cwd().parents]:
    candidates.extend([base, base / BUNDLE_NAME])
if IN_COLAB:
    candidates.extend([
        Path('/content/drive/MyDrive') / BUNDLE_NAME,
        Path('/content/drive/MyDrive/class_A_fermionic_adaptive_circuit') / BUNDLE_NAME,
    ])

seen = set()
BUNDLE_ROOT = None
for candidate in candidates:
    candidate = candidate.resolve()
    if candidate in seen:
        continue
    seen.add(candidate)
    if is_bundle_root(candidate):
        BUNDLE_ROOT = candidate
        break
if BUNDLE_ROOT is None:
    raise FileNotFoundError(f'Could not locate {BUNDLE_NAME} with required src files')

SRC_DIR = BUNDLE_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print(f'[setup] bundle root: {BUNDLE_ROOT}')


## 2. Imports

In [ ]:
import gc
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Image, Markdown, display
from tqdm.auto import tqdm
import torch

from classA_U1FGTN_gpu import classA_U1FGTN_gpu
from strip_entropy_streaming_gpu import (
    HELPER_VERSION,
    StripContourAccumulator,
    entropy_curve_rows_from_accumulator,
    fit_full_x_log_chord,
    rel_to_root,
    write_contour_batch_files,
    write_json_atomic,
)

if not torch.cuda.is_available():
    raise RuntimeError('CUDA is required for this Colab GPU campaign.')

torch.set_grad_enabled(False)
plt.rcParams.update({
    'figure.dpi': 130,
    'savefig.dpi': 180,
    'axes.grid': True,
    'grid.alpha': 0.25,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

print(f'[setup] helper={HELPER_VERSION}')
print(f'[setup] torch={torch.__version__}')
print(f'[setup] cuda device: {torch.cuda.get_device_name(0)}')
free_bytes, total_bytes = torch.cuda.mem_get_info(torch.device('cuda:0'))
print(f'[setup] cuda free/total GiB: {free_bytes/1024**3:.2f}/{total_bytes/1024**3:.2f}')


## 3. Campaign Configuration

In [ ]:
NX = 20
NY = 40
NSHELL = 1
SAMPLES = 100
CYCLES = 100
FIT_AY_MIN = 8
ALPHA_1 = 1.0
ALPHA_2 = 30.0
DW_TRUNCATION = True
INIT_MODE = 'default'
DTYPE = 'complex128'
DEVICE = 'cuda:0'
BACKEND = 'local'
SEQUENCE = 'raster_y'
POSTSELECT = False
PERFECT_CORRECTION = True
N_A = 0.5
SAMPLE_BLOCK_SIZE = 10
SKIP_EXISTING = True
AY_BATCH_SIZE = 4
CYCLE_BATCH_SIZE = 5
AUTOTUNE_FOR_A100_40GB = True
AUTOTUNE_REPEAT = 2
MEMORY_SAFETY_FRACTION = 0.85
EIGH_MEMORY_MULTIPLIER = 8.0
SAMPLE_CHUNK_CANDIDATES = [1, 2, 4, 8, 16]
Y0_CHUNK_CANDIDATES = [1, 2, 4, 6, 8, 12, 16, 24, 32, 'ny']
TARGET_SLOPE_FULL_X = 1.0 / 3.0
SELECTED_PLOT_CYCLES = [0, 5, 10, 20, 50, 100]

if SAMPLES % SAMPLE_BLOCK_SIZE != 0:
    raise ValueError('SAMPLE_BLOCK_SIZE must divide SAMPLES for fixed-width block checkpoints.')

GPU_DATA_ROOT = BUNDLE_ROOT / 'gpu_data'
CAMPAIGN_NAME = 'pure_state_entanglement_slope_vs_cycle'
OUTPUT_ROOT = GPU_DATA_ROOT / CAMPAIGN_NAME
RUN_ID = f'N{NX}x{NY}_nsh{NSHELL}_dwtrunc{int(DW_TRUNCATION)}_C{CYCLES}_S{SAMPLES}'
RUN_ROOT = OUTPUT_ROOT / 'runs' / RUN_ID
CONTOUR_ROOT = RUN_ROOT / 'contour_batches'
FIG_ROOT = RUN_ROOT / 'figures'
BLOCK_ROOT = RUN_ROOT / 'sample_block_checkpoints'
RUN_ROOT.mkdir(parents=True, exist_ok=True)
CONTOUR_ROOT.mkdir(parents=True, exist_ok=True)
FIG_ROOT.mkdir(parents=True, exist_ok=True)
BLOCK_ROOT.mkdir(parents=True, exist_ok=True)
CANONICAL_ENTRY_POINT = 'classA_U1FGTN_gpu.run_markov_circuit'

campaign_config = {
    'Nx': NX,
    'Ny': NY,
    'nshell': NSHELL,
    'samples': SAMPLES,
    'cycles': CYCLES,
    'fit_Ay_min': FIT_AY_MIN,
    'alpha_1': ALPHA_1,
    'alpha_2': ALPHA_2,
    'dw_truncation': DW_TRUNCATION,
    'init_mode': INIT_MODE,
    'dtype': DTYPE,
    'sequence': SEQUENCE,
    'perfect_correction': PERFECT_CORRECTION,
    'postselect': POSTSELECT,
    'canonical_dynamics_entry_point': CANONICAL_ENTRY_POINT,
    'gpu_target': 'A100 40GB',
    'sample_block_size': SAMPLE_BLOCK_SIZE,
    'checkpoint_format': 'full_contour',
}
display(Markdown('### Campaign config'))
display(pd.DataFrame([campaign_config]))


## 4. Shared Helpers

In [ ]:
def show_saved_figure(path: Path, title: str | None = None, note: str | None = None) -> None:
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(path)
    if title:
        display(Markdown(f'**{title}**'))
    if note:
        display(Markdown(note))
    display(Image(filename=str(path)))


def save_dataframe_atomic_csv(df: pd.DataFrame, path: Path) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + '.tmp')
    df.to_csv(tmp, index=False)
    tmp.replace(path)


def save_npz_atomic(path: Path, **arrays: Any) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + '.tmp')
    with tmp.open('wb') as fh:
        np.savez_compressed(fh, **arrays)
    tmp.replace(path)


def update_gpu_data_index(name: str, kind: str, manifest_rel: str) -> None:
    index_path = GPU_DATA_ROOT / 'index.json'
    if index_path.exists():
        payload = json.loads(index_path.read_text())
    else:
        payload = {'bundle_root': str(BUNDLE_ROOT), 'gpu_data_root': str(GPU_DATA_ROOT), 'campaigns': []}
    campaigns = payload.setdefault('campaigns', [])
    campaigns[:] = [item for item in campaigns if item.get('name') != name]
    campaigns.append({'name': name, 'kind': kind, 'manifest': manifest_rel})
    payload['bundle_root'] = str(BUNDLE_ROOT)
    payload['gpu_data_root'] = str(GPU_DATA_ROOT)
    write_json_atomic(index_path, payload)


def block_path_for(block_start: int, block_stop: int) -> Path:
    return BLOCK_ROOT / f'block_{block_start:03d}_{block_stop:03d}.npz'


def make_accumulator(config: dict[str, Any]) -> StripContourAccumulator:
    return StripContourAccumulator(
        nx=NX,
        ny=NY,
        ay_values=range(NY // 2 + 1),
        cycles=range(CYCLES + 1),
        config=config,
    )


def validate_block_checkpoint(path: Path, *, block_count: int) -> None:
    try:
        acc = StripContourAccumulator.load_checkpoint(path, require_contours=True)
    except Exception as exc:
        raise RuntimeError(f'Invalid block checkpoint {path}. Move/delete it before rerunning.') from exc
    if acc.nx != NX or acc.ny != NY or acc.ay_values != list(range(NY // 2 + 1)) or acc.cycles != list(range(CYCLES + 1)):
        raise RuntimeError(f'Block checkpoint {path} has incompatible axes. Move/delete it before rerunning.')
    expected = int(block_count) * NY
    for ay in acc.ay_values:
        count = acc.count[ay]
        if not np.all(count == expected):
            bad_cycles = [int(acc.cycles[idx]) for idx, value in enumerate(count) if int(value) != expected]
            raise RuntimeError(
                f'Block checkpoint {path} is incomplete for Ay={ay}; expected count {expected}. '
                f'Bad cycles start with {bad_cycles[:10]}. Move/delete it before rerunning.'
            )


def validate_final_counts(acc: StripContourAccumulator) -> None:
    expected = SAMPLES * NY
    for ay in acc.ay_values:
        count = acc.count[ay]
        if not np.all(count == expected):
            bad_cycles = [int(acc.cycles[idx]) for idx, value in enumerate(count) if int(value) != expected]
            raise RuntimeError(
                f'Merged accumulator is incomplete for Ay={ay}; expected count {expected}. '
                f'Bad cycles start with {bad_cycles[:10]}.'
            )


def plot_log_chord_fit(entropy_curves: pd.DataFrame, fit_row: dict, *, fig_path: Path, title: str) -> None:
    fig, ax = plt.subplots(figsize=(7.2, 4.6), constrained_layout=True)
    sub = entropy_curves[(entropy_curves['cycle'] == int(fit_row['cycle'])) & (entropy_curves['Ay'] > 0)].sort_values('Ay')
    x_all = sub['log_sin_pi_Ay_over_Ny'].to_numpy(dtype=np.float64)
    y_all = sub['entropy_mean'].to_numpy(dtype=np.float64)
    yerr = sub['entropy_sem'].to_numpy(dtype=np.float64)
    x_band_min = float(fit_row['fit_x_min'])
    x_band_max = float(fit_row['fit_x_max'])
    x_line = np.linspace(float(np.min(x_all)), float(np.max(x_all)), 300)
    y_line = float(fit_row['slope']) * x_line + float(fit_row['intercept'])
    ax.axvspan(x_band_min, x_band_max, color='0.5', alpha=0.18, label=f"fit Ay={int(fit_row['Ay_fit_min'])}..{int(fit_row['Ay_fit_max'])}")
    ax.errorbar(x_all, y_all, yerr=yerr, marker='o', ms=3.5, lw=1.3, capsize=2, color='tab:blue', label='full_x S(Ay) data')
    ax.plot(x_line, y_line, color='black', ls='--', lw=1.5, label=f"fit m={float(fit_row['slope']):.4f}; r2={float(fit_row['r2']):.4f}")
    ax.set_title(title)
    ax.set_xlabel(r'$\log[\sin(\pi A_y/N_y)]$')
    ax.set_ylabel(r'full_x $S(A_y)$')
    ax.legend(frameon=True, fontsize=8)
    fig.savefig(fig_path)
    plt.close(fig)


## 5. Run Blocks, Merge, And Save Products

In [ ]:
summary_path = RUN_ROOT / 'run_summary.json'
fit_path = RUN_ROOT / 'slope_vs_cycle.csv'
entropy_path = RUN_ROOT / 'entropy_curves.csv'
full_checkpoint_path = RUN_ROOT / 'accumulator_checkpoint_full.npz'
legacy_checkpoint_path = RUN_ROOT / 'accumulator_checkpoint.npz'

fig_slope = FIG_ROOT / 'slope_vs_cycle.png'
fig_r2 = FIG_ROOT / 'r2_vs_cycle.png'
fig_selected = FIG_ROOT / 'entropy_vs_log_chord_selected_cycles.png'
fig_heat = FIG_ROOT / 'entropy_heatmap_cycle_Ay.png'

summary = None
if SKIP_EXISTING and summary_path.exists() and fit_path.exists() and entropy_path.exists():
    existing_summary = json.loads(summary_path.read_text())
    if existing_summary.get('status') == 'complete':
        print(f'[skip complete run] {RUN_ID}')
        summary = existing_summary
        entropy_curves = pd.read_csv(entropy_path)
        slope_vs_cycle = pd.read_csv(fit_path)

if summary is None:
    block_records = []
    for block_start in tqdm(range(0, SAMPLES, SAMPLE_BLOCK_SIZE), desc='sample blocks', unit='block'):
        block_stop = min(SAMPLES, block_start + SAMPLE_BLOCK_SIZE)
        block_count = block_stop - block_start
        block_path = block_path_for(block_start, block_stop)
        if block_path.exists():
            validate_block_checkpoint(block_path, block_count=block_count)
            print(f'[skip block] {block_start}:{block_stop}')
            block_records.append({
                'block_start': int(block_start),
                'block_stop': int(block_stop),
                'checkpoint_relative': rel_to_root(block_path, GPU_DATA_ROOT),
                'status': 'existing',
            })
            continue

        print(f'[run block] {block_start}:{block_stop}')
        block_config = {
            **campaign_config,
            'run_id': RUN_ID,
            'Ay_values': list(range(NY // 2 + 1)),
            'block_start': int(block_start),
            'block_stop': int(block_stop),
            'block_count': int(block_count),
        }
        block_accumulator = make_accumulator(block_config)

        def cycle_observer(*, cycle, G, batch_index, batch_start, batch_count):
            print(
                f'[observer] block={block_start}:{block_stop} cycle={cycle} '
                f'batch={batch_index} samples {batch_start}:{batch_start + batch_count}',
                flush=True,
            )
            block_accumulator.update(
                cycle=cycle,
                G=G,
                eps=1e-12,
                autotune=AUTOTUNE_FOR_A100_40GB,
                sample_chunk_candidates=SAMPLE_CHUNK_CANDIDATES,
                y0_chunk_candidates=Y0_CHUNK_CANDIDATES,
                autotune_repeat=AUTOTUNE_REPEAT,
                memory_safety_fraction=MEMORY_SAFETY_FRACTION,
                eigh_memory_multiplier=EIGH_MEMORY_MULTIPLIER,
            )

        model = classA_U1FGTN_gpu(
            Nx=NX,
            Ny=NY,
            DW=True,
            nshell=NSHELL,
            filling_frac=0.5,
            alpha_1=ALPHA_1,
            alpha_2=ALPHA_2,
            trial_orbitals='X',
            dw_truncation=DW_TRUNCATION,
            triv_region_local_mode=False,
            device=DEVICE,
            dtype=DTYPE,
            backend=BACKEND,
        )
        result = model.run_markov_circuit(
            G_history=False,
            progress=False,
            cycles=CYCLES,
            postselect=POSTSELECT,
            perfect_correction=PERFECT_CORRECTION,
            samples=block_count,
            init_mode=INIT_MODE,
            save=False,
            n_a=N_A,
            sequence=SEQUENCE,
            batch_size=block_count,
            return_data=False,
            cycle_observer=cycle_observer,
        )
        block_accumulator.save_checkpoint(block_path, include_contours=True)
        validate_block_checkpoint(block_path, block_count=block_count)
        block_records.append({
            'block_start': int(block_start),
            'block_stop': int(block_stop),
            'checkpoint_relative': rel_to_root(block_path, GPU_DATA_ROOT),
            'status': 'created',
            'markov_result': result,
        })
        del model, block_accumulator
        gc.collect()
        torch.cuda.empty_cache()

    accumulator = make_accumulator({
        **campaign_config,
        'run_id': RUN_ID,
        'Ay_values': list(range(NY // 2 + 1)),
        'merged_from_blocks': True,
    })
    for block_start in range(0, SAMPLES, SAMPLE_BLOCK_SIZE):
        block_stop = min(SAMPLES, block_start + SAMPLE_BLOCK_SIZE)
        accumulator.merge_checkpoint(block_path_for(block_start, block_stop), require_contours=True)
    validate_final_counts(accumulator)
    accumulator.save_checkpoint(full_checkpoint_path, include_contours=True)

    batch_records = write_contour_batch_files(
        accumulator=accumulator,
        output_dir=CONTOUR_ROOT,
        ay_batch_size=AY_BATCH_SIZE,
        cycle_batch_size=CYCLE_BATCH_SIZE,
        config={**campaign_config, 'run_id': RUN_ID},
    )
    entropy_curves = pd.DataFrame(entropy_curve_rows_from_accumulator(accumulator)).sort_values(['cycle', 'Ay']).reset_index(drop=True)
    save_dataframe_atomic_csv(entropy_curves, entropy_path)

    slope_rows = []
    for cycle in range(CYCLES + 1):
        slope_rows.append(fit_full_x_log_chord(
            entropy_curves,
            nx=NX,
            ny=NY,
            nshell=NSHELL,
            cycle=cycle,
            fit_ay_min=FIT_AY_MIN,
            target_slope=TARGET_SLOPE_FULL_X,
        ))
    slope_vs_cycle = pd.DataFrame(slope_rows).sort_values('cycle').reset_index(drop=True)
    save_dataframe_atomic_csv(slope_vs_cycle, fit_path)
    save_npz_atomic(
        RUN_ROOT / 'slope_vs_cycle.npz',
        cycle=slope_vs_cycle['cycle'].to_numpy(dtype=np.int64),
        slope=slope_vs_cycle['slope'].to_numpy(dtype=np.float64),
        slope_err=slope_vs_cycle['slope_err'].to_numpy(dtype=np.float64),
        r2=slope_vs_cycle['r2'].to_numpy(dtype=np.float64),
        config_json=np.asarray(json.dumps(campaign_config, sort_keys=True)),
    )

    summary = {
        'status': 'complete',
        'run_id': RUN_ID,
        'config': campaign_config,
        'run_root': str(RUN_ROOT),
        'run_root_relative': rel_to_root(RUN_ROOT, GPU_DATA_ROOT),
        'block_checkpoints_relative': [record['checkpoint_relative'] for record in block_records],
        'block_records': block_records,
        'contour_batches_relative': [rel_to_root(record['filename'], GPU_DATA_ROOT) for record in batch_records],
        'entropy_curves_relative': rel_to_root(entropy_path, GPU_DATA_ROOT),
        'slope_vs_cycle_relative': rel_to_root(fit_path, GPU_DATA_ROOT),
        'checkpoint_relative': rel_to_root(full_checkpoint_path, GPU_DATA_ROOT),
        'legacy_partial_checkpoint_relative': rel_to_root(legacy_checkpoint_path, GPU_DATA_ROOT) if legacy_checkpoint_path.exists() else None,
        'canonical_dynamics_entry_point': CANONICAL_ENTRY_POINT,
        'batching': {
            'sample_block_size': SAMPLE_BLOCK_SIZE,
            'contour_autotune_enabled': AUTOTUNE_FOR_A100_40GB,
            'sample_chunk_candidates': SAMPLE_CHUNK_CANDIDATES,
            'y0_chunk_candidates': Y0_CHUNK_CANDIDATES,
            'memory_safety_fraction': MEMORY_SAFETY_FRACTION,
            'eigh_memory_multiplier': EIGH_MEMORY_MULTIPLIER,
        },
        'created_unix': time.time(),
    }
    write_json_atomic(summary_path, summary)

display(Markdown('### Slope vs cycle rows'))
display(slope_vs_cycle)


## 6. Figures And Manifest

In [ ]:
if not all(path.exists() for path in [fig_slope, fig_r2, fig_selected, fig_heat]):
    fig, ax = plt.subplots(figsize=(7.4, 4.6), constrained_layout=True)
    ax.errorbar(slope_vs_cycle['cycle'], slope_vs_cycle['slope'], yerr=slope_vs_cycle['slope_err'], marker='o', ms=2.5, lw=1.2, capsize=2)
    ax.axhline(TARGET_SLOPE_FULL_X, color='black', ls='--', lw=1.0, label='target 1/3')
    ax.set_xlabel('cycle')
    ax.set_ylabel('full_x log-chord slope')
    ax.set_title(f'N{NX}x{NY} slope vs cycle')
    ax.legend(frameon=True)
    fig.savefig(fig_slope)
    plt.close(fig)

    fig, ax = plt.subplots(figsize=(7.4, 4.2), constrained_layout=True)
    ax.plot(slope_vs_cycle['cycle'], slope_vs_cycle['r2'], marker='o', ms=2.5, lw=1.2)
    ax.set_xlabel('cycle')
    ax.set_ylabel('R^2')
    ax.set_title('Log-chord fit R^2 vs cycle')
    fig.savefig(fig_r2)
    plt.close(fig)

    fig, axes = plt.subplots(2, 3, figsize=(14.0, 7.6), constrained_layout=True, sharey=True)
    axes = axes.ravel()
    for ax, cycle in zip(axes, SELECTED_PLOT_CYCLES):
        fit_row = slope_vs_cycle[slope_vs_cycle['cycle'] == cycle].iloc[0].to_dict()
        sub = entropy_curves[(entropy_curves['cycle'] == cycle) & (entropy_curves['Ay'] > 0)].sort_values('Ay')
        x_all = sub['log_sin_pi_Ay_over_Ny'].to_numpy(dtype=np.float64)
        y_all = sub['entropy_mean'].to_numpy(dtype=np.float64)
        yerr = sub['entropy_sem'].to_numpy(dtype=np.float64)
        x_line = np.linspace(float(np.min(x_all)), float(np.max(x_all)), 300)
        y_line = float(fit_row['slope']) * x_line + float(fit_row['intercept'])
        ax.axvspan(float(fit_row['fit_x_min']), float(fit_row['fit_x_max']), color='0.5', alpha=0.18)
        ax.errorbar(x_all, y_all, yerr=yerr, marker='o', ms=3, lw=1.1, capsize=2)
        ax.plot(x_line, y_line, color='black', ls='--', lw=1.3, label=f"m={float(fit_row['slope']):.3f}")
        ax.set_title(f'cycle {cycle}')
        ax.set_xlabel(r'$\log[\sin(\pi A_y/N_y)]$')
        ax.set_ylabel(r'full_x $S(A_y)$')
        ax.legend(frameon=True, fontsize=8)
    fig.savefig(fig_selected)
    plt.close(fig)

    pivot = entropy_curves.pivot_table(index='cycle', columns='Ay', values='entropy_mean', aggfunc='first').sort_index()
    fig, ax = plt.subplots(figsize=(8.0, 4.8), constrained_layout=True)
    im = ax.imshow(
        pivot.to_numpy(dtype=np.float64),
        origin='lower',
        aspect='auto',
        cmap='Blues',
        extent=[pivot.columns.min(), pivot.columns.max(), pivot.index.min(), pivot.index.max()],
    )
    ax.set_xlabel('Ay')
    ax.set_ylabel('cycle')
    ax.set_title('Full-x strip entropy S(Ay, cycle)')
    fig.colorbar(im, ax=ax, label='S(Ay)')
    fig.savefig(fig_heat)
    plt.close(fig)

manifest = {
    'campaign_name': CAMPAIGN_NAME,
    'campaign_config': campaign_config,
    'canonical_dynamics_entry_point': CANONICAL_ENTRY_POINT,
    'helper_version': HELPER_VERSION,
    'bundle_root': str(BUNDLE_ROOT),
    'gpu_data_root': str(GPU_DATA_ROOT),
    'output_root': str(OUTPUT_ROOT),
    'output_root_relative': rel_to_root(OUTPUT_ROOT, GPU_DATA_ROOT),
    'run_summary_relative': rel_to_root(summary_path, GPU_DATA_ROOT),
    'figures_relative': [rel_to_root(p, GPU_DATA_ROOT) for p in [fig_slope, fig_r2, fig_selected, fig_heat]],
    'batch_saving': {'Ay_batch_size': AY_BATCH_SIZE, 'cycle_batch_size': CYCLE_BATCH_SIZE},
    'sample_block_size': SAMPLE_BLOCK_SIZE,
}
manifest_path = OUTPUT_ROOT / 'campaign_manifest.json'
write_json_atomic(manifest_path, manifest)
update_gpu_data_index(CAMPAIGN_NAME, 'pure_state_entanglement_slope_dynamics', rel_to_root(manifest_path, GPU_DATA_ROOT))

for p, title in [(fig_slope, 'Slope vs cycle'), (fig_r2, 'R2 vs cycle'), (fig_selected, 'Selected-cycle log-chord fits'), (fig_heat, 'Entropy heatmap')]:
    show_saved_figure(p, title)
print(f'[done] campaign manifest: {manifest_path}')


## 7. Disconnect Runtime

In [ ]:
if IN_COLAB:
    from google.colab import runtime  # type: ignore
    runtime.unassign()
else:
    print('[done] not running in Colab; runtime disconnect skipped')
